# 🎤 DSM-ASR v4: Instruction Fine-Tuning

**Mimi audio tokens + Qwen3-0.6B = ASR**

```
[Instruction] [audio_emb_1...audio_emb_T] [Separator] [Transcription]
```

- No timestamps needed — simple & fast
- Audio adapter bridges Mimi → Qwen3 embedding space
- Prints predictions during training to track progress

**Runtime**: A100 40GB

## 1. Setup

In [ ]:
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}, GPU: {torch.cuda.get_device_name(0)}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB')

In [ ]:
REPO_URL = "https://github.com/AhPro7/DSM.git"
!rm -rf /content/DSM
!git clone {REPO_URL} /content/DSM
%cd /content/DSM

In [ ]:
!pip install -q torch transformers>=4.51.0 datasets[audio] accelerate \
    soundfile librosa tqdm jiwer wandb

In [ ]:
from huggingface_hub import login
login()

## 2. Preprocess (Mimi encoding only — fast!)

In [ ]:
# Quick test with 20 samples
!python data/prepare_data.py --max_samples 20 --batch_size 8

In [ ]:
import json
with open('preprocessed_data/manifest.json') as f:
    m = json.load(f)
print(f'OK: {m["total_processed"]}, Errors: {m["total_errors"]}')
if m['samples']:
    print(f'Sample: {m["samples"][0]["num_frames"]} frames, {m["samples"][0]["text"][:60]}')

In [ ]:
# FULL dataset (uncomment when ready)
# !rm -rf preprocessed_data
# !python data/prepare_data.py --batch_size 16

## 3. Sanity Checks

In [ ]:
!python data/dataset.py

In [ ]:
!python model/dsm_asr.py

## 4. Train
Prints **sample predictions with WER** every 50 steps + at each epoch end

In [ ]:
# Quick test (10 steps)
!rm -rf output
!python train.py --max_steps 10 --batch_size 2 --max_samples 20

In [ ]:
# Full training
!rm -rf output
!python train.py --batch_size 4 --num_epochs 15

In [ ]:
import json, matplotlib.pyplot as plt
try:
    with open('output/training_log.json') as f: log = json.load(f)
    plt.figure(figsize=(10,4))
    plt.plot([e['step'] for e in log], [e['loss'] for e in log])
    plt.xlabel('Step'); plt.ylabel('Loss'); plt.title('Training Loss')
    plt.grid(True, alpha=0.3); plt.show()
except: print('No log yet')

## 5. Evaluate

In [ ]:
!python evaluate.py --checkpoint output/best --output output/eval.json

## 6. Inference

In [ ]:
from google.colab import files
uploaded = files.upload()
f = list(uploaded.keys())[0]
!python inference.py --checkpoint output/best --audio {f}

## 7. Save Model

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
repo_id = 'nadsoft/dsm-asr-arabic'
api.create_repo(repo_id, exist_ok=True)
api.upload_folder(folder_path='output/best', repo_id=repo_id)
print(f'✅ Uploaded to https://huggingface.co/{repo_id}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r output/best /content/drive/MyDrive/dsm_asr_v4/
print('✅ Saved to Drive')